<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자연어 처리와 LLM · 09. RAG

## Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks

- **원 논문:** [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)
- **저자 / 발표:** Patrick Lewis et al. · NeurIPS 2020 (2020)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch, 확률·선형대수·sequence model의 기초
- **로컬 학습 데이터:** `data/field_curriculum/nlp_corpus.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** retriever distribution과 RAG-Sequence marginalization을 재현한다.

**축소하거나 생략한 부분:** DPR+BART+Wikipedia 21M passages 대신 TF-IDF와 로컬 5문서를 사용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2.1 Eq. (1): pη(z|x) | Task 1 softmax similarity | probability sum |
| §2.2 Eq. (2): RAG-Sequence | Task 2 marginalization | finite NLL |
| §2.2 Eq. (3): RAG-Token | Task 2 설명/비교 | per-token distinction |
| Fig. 1: retriever+generator | Task 3 | recall@1/cited answer |

## 핵심 아이디어와 수식

$$p_{RAG\text{-}Seq}(y\mid x)=\sum_{z\in top\text{-}k}p_{\eta}(z\mid x)p_{\theta}(y\mid x,z)$$

**기호 해설:** z는 문서, p_eta는 retriever, p_theta는 document-conditioned generator likelihood다.

**코드 대응:** Task 1이 TF-IDF retriever, Task 2가 RAGSequence class, Task 3이 NLL/recall을 구현한다.

**입력과 출력 shape:** query vectors [3,V], document vectors [5,V] → p(z|x) [3,5] → answer prob [3].

**포트폴리오 구현 범위:** neural DPR/BART 대신 결정적 surrogate를 쓰지만 latent-document marginalization은 정확하다.

각 단계는 데이터 전처리와 shape 확인부터 논문 고유 class, `forward`, 목적함수,
`train_step`, 평가까지 이어진다. 핵심 수식을 한 번의 고수준 호출로 감추지 않고
중간 tensor를 확인할 수 있게 분리한다.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import math
import re
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/nlp_corpus.json")
corpus = json.loads(dataset_path.read_text(encoding="utf-8"))
sentences = corpus["sentences"]
translation_pairs = corpus["translation_pairs"]
documents = corpus["documents"]
queries = corpus["queries"]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<mask>", "<sentinel>"]
sentence_words = {word.lower() for sentence in sentences for word in sentence.split()}
translation_words = {
    word.lower()
    for pair in translation_pairs
    for side in ("source", "target")
    for word in pair[side].split()
}
words = sorted(sentence_words | translation_words)
itos = SPECIAL_TOKENS + words
stoi = {word: index for index, word in enumerate(itos)}
PAD, BOS, EOS, MASK, SENTINEL = range(5)
VOCAB_SIZE = len(itos)


def encode(text, length=4):
    token_ids = [stoi[word.lower()] for word in text.split()]
    token_ids = token_ids[:length]
    return token_ids + [PAD] * (length - len(token_ids))


sentence_ids = torch.tensor([encode(text) for text in sentences])
sentiment_labels = torch.tensor(corpus["sentiment_labels"], dtype=torch.long)
source_ids = torch.tensor([encode(pair["source"], 3) for pair in translation_pairs])
target_ids = torch.tensor([encode(pair["target"], 3) for pair in translation_pairs])
sentence_ids, sentiment_labels = ACCELERATOR.move(
    sentence_ids,
    sentiment_labels,
)
source_ids, target_ids = ACCELERATOR.move(source_ids, target_ids)
assert sentence_ids.shape == (360, 4)
assert source_ids.shape == target_ids.shape == (60, 3)
print(ACCELERATOR.summary())
print(dataset_path.name, f"sentences={len(sentences)}, vocab={VOCAB_SIZE}")

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.1 Eq. (1): pη(z|x)
- **노트북에서 구현할 부분:** Task 1 softmax similarity
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** probability sum

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
@dataclass
class RAGConfig:
    generator_match_probability: float = 0.9
    generator_miss_probability: float = 0.05
    top_k: int = 3


def terms(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def count_vector(text, vocabulary, term_to_id):
    vector = torch.zeros(len(vocabulary))
    for term, count in Counter(terms(text)).items():
        vector[term_to_id[term]] = count
    return vector


def retrieval_distribution(query_vectors, document_vectors):
    similarity = query_vectors @ document_vectors.T
    return similarity.softmax(dim=1), similarity


config = RAGConfig()
vocabulary = sorted(
    {term for document in documents for term in terms(document["text"])}
    | {term for query in queries for term in terms(query["query"])}
)
term_to_id = {term: index for index, term in enumerate(vocabulary)}
document_tf = torch.stack(
    [count_vector(document["text"], vocabulary, term_to_id) for document in documents]
)
document_frequency = (document_tf > 0).sum(dim=0)
inverse_frequency = torch.log((1 + len(documents)) / (1 + document_frequency)) + 1
document_vectors = F.normalize(document_tf * inverse_frequency, dim=1)
query_tf = torch.stack(
    [count_vector(query["query"], vocabulary, term_to_id) for query in queries]
)
query_vectors = F.normalize(query_tf * inverse_frequency, dim=1)
retrieval_probability, similarity = retrieval_distribution(
    query_vectors,
    document_vectors,
)
assert retrieval_probability.shape == (3, 5)
assert torch.allclose(retrieval_probability.sum(1), torch.ones(3))

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.2 Eq. (2): RAG-Sequence
- **노트북에서 구현할 부분:** Task 2 marginalization
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** finite NLL

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
class RAGSequence:
    def __init__(self, documents, config):
        self.documents = documents
        self.document_ids = [document["id"] for document in documents]
        self.config = config

    def generator_likelihood(self, queries, document_ids):
        likelihood = torch.empty(len(queries), len(document_ids))
        for row, query in enumerate(queries):
            for column, document_id in enumerate(document_ids):
                if document_id == query["answer_id"]:
                    value = self.config.generator_match_probability
                else:
                    value = self.config.generator_miss_probability
                likelihood[row, column] = value
        return likelihood

    def forward(self, retrieval_probability, queries):
        generator = self.generator_likelihood(
            queries,
            self.document_ids,
        )
        return (retrieval_probability * generator).sum(dim=1)

    def grounded_answer(self, document_index):
        document = self.documents[document_index]
        return f"[{document['id']}] {document['text']}"


model = RAGSequence(documents, config)
sequence_probability = model.forward(retrieval_probability, queries)
assert sequence_probability.shape == (3,)
assert bool((sequence_probability > 0).all())

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** Fig. 1: retriever+generator
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** recall@1/cited answer

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
def rag_sequence_loss(sequence_probability):
    return -sequence_probability.clamp_min(1e-8).log().mean()


def evaluate_retrieval(similarity, documents, queries):
    top_indices = similarity.argmax(dim=1)
    document_ids = [documents[index]["id"] for index in top_indices.tolist()]
    matches = [
        predicted == query["answer_id"]
        for predicted, query in zip(document_ids, queries)
    ]
    return sum(matches) / len(matches), top_indices


def build_grounded_answers(model, top_indices):
    return [
        model.grounded_answer(document_index) for document_index in top_indices.tolist()
    ]


loss = rag_sequence_loss(sequence_probability)
recall, top_indices = evaluate_retrieval(similarity, documents, queries)
answers = build_grounded_answers(model, top_indices)
assert loss.isfinite()
assert recall >= 2 / 3
assert all(answer.startswith("[") for answer in answers)
plt.figure(figsize=(7, 3))
plt.imshow(similarity)
plt.xticks(range(len(documents)), [item["id"] for item in documents], rotation=30)
plt.yticks(range(len(queries)), [f"q{index + 1}" for index in range(len(queries))])
plt.colorbar(label="cosine")
plt.title(f"RAG retrieval recall@1={recall:.1%}")
plt.tight_layout()
plt.show()
print("\n".join(answers))

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> DPR+BART+Wikipedia 21M passages 대신 TF-IDF와 로컬 5문서를 사용한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.